<a href="https://colab.research.google.com/github/nphngan282006-hub/KPW/blob/main/run_phobert_kfold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhoBERT K-fold cho ABSA khách sạn (Colab GPU)
Dùng lại `kfold_visobert.py` (đã thêm tham số `text_fn` tách từ và chế độ tất định). **Hạt giống cố định `SEED = 42`**.
PhoBERT cần văn bản **đã tách từ**, nên dùng `pyvi` (thuần Python, kết quả ổn định).
Chuẩn bị: Runtime → GPU (T4). Upload `kfold_visobert.py`, `train.csv`, `test_blind.csv`, `visobert_test_ids.csv` vào thư mục làm việc Colab.

In [ ]:
!pip -q install transformers pyvi iterative-stratification
import torch, os, random, numpy as np
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
print(torch.cuda.get_device_name(0))

Tesla T4


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/absa/outputs"      # kết quả lưu ở Drive để không mất khi Colab ngắt
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
from pyvi import ViTokenizer
from kfold_visobert import run_kfold
segment = lambda t: ViTokenizer.tokenize(t)          # "khách sạn rất sạch" -> "khách_sạn rất sạch"
print(segment("Nhân viên thân thiện, phòng sạch sẽ gần bãi biển"))

Nhân_viên thân_thiện , phòng sạch_sẽ gần bãi biển


In [ ]:
# Chạy thử nhanh (2 fold, 1 epoch) để kiểm tra không lỗi trước khi chạy thật
_ = run_kfold("train.csv", "visobert_test_ids.csv", None, out_dir="outputs_test", tag="smoke_pho",
              model="vinai/phobert-base-v2", n_folds=2, epochs=1, text_fn=segment, max_len=256, save_weights=False, seed=SEED)

Thiết bị: cuda (Tesla T4)
Dev (K-fold): 3750 | holdout: 938


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Chia fold bằng: iterative-stratification


... (6 outputs left out)

  Fold 0 | epoch 1/1 | loss 1.0502 | val ABSA-F1 0.4525 | 21s


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 1 | epoch 1/1 | loss 1.0456 | val ABSA-F1 0.4785 | 19s

OOF ABSA-F1 (ngưỡng 0,5): 0.4671 | sau chỉnh ngưỡng: 0.5383 (lạc quan vì ngưỡng chọn trên chính OOF)
HOLDOUT ABSA-F1: ngưỡng 0,5 = 0.4594 | ngưỡng chỉnh từ OOF = 0.5537   (TF-IDF cùng tập: 0,764)
          nhan  so_mau_that  so_du_doan     P     R    F1
nhan_vien_khen          494         467 0.887 0.838 0.862
 nhan_vien_che          120         110 0.655 0.600 0.626
  sach_se_khen          301         299 0.689 0.684 0.687
   sach_se_che           89          94 0.511 0.539 0.525
   vi_tri_khen          307         324 0.716 0.756 0.735
    vi_tri_che           53           0 0.000 0.000 0.000
    phong_khen          311         372 0.599 0.717 0.653
     phong_che          254         247 0.619 0.602 0.611
  gia_tri_khen          131          57 0.842 0.366 0.511
   gia_tri_che           52          29 0.655 0.365 0.469
  an_uong_khen          173         137 0.796 0.630 0.703
   an_uong_che          130          75 0.653

In [ ]:
# Chạy thật. Muốn chạy lại phải đổi tag (cùng tag sẽ bị bỏ qua các fold đã xong).
summary = run_kfold("train.csv", "visobert_test_ids.csv", "test_blind.csv", out_dir=OUT_DIR, tag="phobert_kfold",
                    model="vinai/phobert-base-v2", n_folds=5, epochs=12, batch_size=16, lr=3e-5, head_lr=2e-4,
                    llrd=0.9, weight_decay=0.01, warmup=0.1, patience=3, max_len=256, seed=SEED,
                    text_fn=segment, save_weights=True)

Thiết bị: cuda (Tesla T4)
Dev (K-fold): 3750 | holdout: 938
Chia fold bằng: iterative-stratification
Fold 0: đã có kết quả, bỏ qua.
Fold 1: đã có kết quả, bỏ qua.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 2 | epoch 1/12 | loss 1.1318 | val ABSA-F1 0.4881 | 31s
  Fold 2 | epoch 2/12 | loss 0.6914 | val ABSA-F1 0.7319 | 32s
  Fold 2 | epoch 3/12 | loss 0.4892 | val ABSA-F1 0.8045 | 31s
  Fold 2 | epoch 4/12 | loss 0.3722 | val ABSA-F1 0.8403 | 31s
  Fold 2 | epoch 5/12 | loss 0.2897 | val ABSA-F1 0.8611 | 31s
  Fold 2 | epoch 6/12 | loss 0.2327 | val ABSA-F1 0.8694 | 32s
  Fold 2 | epoch 7/12 | loss 0.1877 | val ABSA-F1 0.8725 | 31s
  Fold 2 | epoch 8/12 | loss 0.1546 | val ABSA-F1 0.8783 | 31s
  Fold 2 | epoch 9/12 | loss 0.1267 | val ABSA-F1 0.8812 | 32s
  Fold 2 | epoch 10/12 | loss 0.1102 | val ABSA-F1 0.8803 | 32s
  Fold 2 | epoch 11/12 | loss 0.0985 | val ABSA-F1 0.8824 | 31s
  Fold 2 | epoch 12/12 | loss 0.0907 | val ABSA-F1 0.8830 | 32s


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 3 | epoch 1/12 | loss 1.1379 | val ABSA-F1 0.4987 | 31s
  Fold 3 | epoch 2/12 | loss 0.7062 | val ABSA-F1 0.7170 | 31s
  Fold 3 | epoch 3/12 | loss 0.5101 | val ABSA-F1 0.7753 | 30s
  Fold 3 | epoch 4/12 | loss 0.3922 | val ABSA-F1 0.8192 | 31s
  Fold 3 | epoch 5/12 | loss 0.3108 | val ABSA-F1 0.8435 | 30s
  Fold 3 | epoch 6/12 | loss 0.2489 | val ABSA-F1 0.8605 | 31s
  Fold 3 | epoch 7/12 | loss 0.2008 | val ABSA-F1 0.8727 | 31s
  Fold 3 | epoch 8/12 | loss 0.1652 | val ABSA-F1 0.8665 | 30s
  Fold 3 | epoch 9/12 | loss 0.1381 | val ABSA-F1 0.8807 | 31s
  Fold 3 | epoch 10/12 | loss 0.1187 | val ABSA-F1 0.8813 | 31s
  Fold 3 | epoch 11/12 | loss 0.1046 | val ABSA-F1 0.8858 | 31s
  Fold 3 | epoch 12/12 | loss 0.0970 | val ABSA-F1 0.8867 | 31s


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 4 | epoch 1/12 | loss 1.1404 | val ABSA-F1 0.5893 | 32s
  Fold 4 | epoch 2/12 | loss 0.6892 | val ABSA-F1 0.7179 | 32s
  Fold 4 | epoch 3/12 | loss 0.4825 | val ABSA-F1 0.8088 | 31s
  Fold 4 | epoch 4/12 | loss 0.3671 | val ABSA-F1 0.8424 | 32s
  Fold 4 | epoch 5/12 | loss 0.2872 | val ABSA-F1 0.8572 | 32s
  Fold 4 | epoch 6/12 | loss 0.2263 | val ABSA-F1 0.8650 | 32s
  Fold 4 | epoch 7/12 | loss 0.1827 | val ABSA-F1 0.8695 | 32s
  Fold 4 | epoch 8/12 | loss 0.1498 | val ABSA-F1 0.8738 | 32s
  Fold 4 | epoch 9/12 | loss 0.1262 | val ABSA-F1 0.8685 | 32s
  Fold 4 | epoch 10/12 | loss 0.1071 | val ABSA-F1 0.8770 | 32s
  Fold 4 | epoch 11/12 | loss 0.0947 | val ABSA-F1 0.8737 | 31s
  Fold 4 | epoch 12/12 | loss 0.0878 | val ABSA-F1 0.8736 | 31s

OOF ABSA-F1 (ngưỡng 0,5): 0.8843 | sau chỉnh ngưỡng: 0.8843 (lạc quan vì ngưỡng chọn trên chính OOF)
HOLDOUT ABSA-F1: ngưỡng 0,5 = 0.8869 | ngưỡng chỉnh từ OOF = 0.8856   (TF-IDF cùng tập: 0,764)
          nhan  so_mau_that  so_du_doan     

Tải thư mục `outputs` (các file `phobert_kfold_*_probs.npy`) về cùng thư mục với `blend_final.py`, rồi chạy `python blend_final.py check` và `python blend_final.py submit`.
Nếu còn thời gian GPU, chạy thêm một lần với `seed=7` và `tag="phobert_seed7"` (thêm tag đó vào `TAGS` trong `blend_final.py`): ghép nhiều lần chạy là cách tăng điểm ổn định nhất đã đo với ViSoBERT.

In [ ]:
# ===== Chạy lần 2: seed 7 (cùng cấu hình, chỉ đổi seed và tag) =====
SEED2 = 7
os.environ["PYTHONHASHSEED"] = str(SEED2)
random.seed(SEED2); np.random.seed(SEED2); torch.manual_seed(SEED2); torch.cuda.manual_seed_all(SEED2)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

summary2 = run_kfold("train.csv", "visobert_test_ids.csv", "test_blind.csv", out_dir=OUT_DIR,
                     tag="phobert_seed7",                      # tag mới nên không bỏ qua fold
                     model="vinai/phobert-base-v2", n_folds=5, epochs=12, batch_size=16,
                     lr=3e-5, head_lr=2e-4, llrd=0.9, weight_decay=0.01, warmup=0.1,
                     patience=3, max_len=256, seed=SEED2,
                     text_fn=segment, save_weights=True)

Thiết bị: cuda (Tesla T4)
Dev (K-fold): 3750 | holdout: 938
Chia fold bằng: iterative-stratification


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 0 | epoch 1/12 | loss 1.1289 | val ABSA-F1 0.5013 | 32s
  Fold 0 | epoch 2/12 | loss 0.6964 | val ABSA-F1 0.7283 | 32s
  Fold 0 | epoch 3/12 | loss 0.4999 | val ABSA-F1 0.8130 | 31s
  Fold 0 | epoch 4/12 | loss 0.3816 | val ABSA-F1 0.8479 | 31s
  Fold 0 | epoch 5/12 | loss 0.3030 | val ABSA-F1 0.8579 | 31s
  Fold 0 | epoch 6/12 | loss 0.2455 | val ABSA-F1 0.8675 | 31s
  Fold 0 | epoch 7/12 | loss 0.1986 | val ABSA-F1 0.8770 | 31s
  Fold 0 | epoch 8/12 | loss 0.1652 | val ABSA-F1 0.8794 | 31s
  Fold 0 | epoch 9/12 | loss 0.1395 | val ABSA-F1 0.8828 | 31s
  Fold 0 | epoch 10/12 | loss 0.1179 | val ABSA-F1 0.8815 | 31s
  Fold 0 | epoch 11/12 | loss 0.1060 | val ABSA-F1 0.8857 | 31s
  Fold 0 | epoch 12/12 | loss 0.0973 | val ABSA-F1 0.8881 | 31s


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (6 outputs left out)

  Fold 3 | epoch 1/12 | loss 1.1316 | val ABSA-F1 0.5485 | 32s
  Fold 3 | epoch 2/12 | loss 0.6955 | val ABSA-F1 0.7114 | 32s
  Fold 3 | epoch 3/12 | loss 0.4963 | val ABSA-F1 0.8068 | 31s
  Fold 3 | epoch 4/12 | loss 0.3805 | val ABSA-F1 0.8382 | 31s
  Fold 3 | epoch 5/12 | loss 0.2971 | val ABSA-F1 0.8607 | 31s
  Fold 3 | epoch 6/12 | loss 0.2369 | val ABSA-F1 0.8731 | 31s
  Fold 3 | epoch 7/12 | loss 0.1899 | val ABSA-F1 0.8820 | 31s
  Fold 3 | epoch 8/12 | loss 0.1554 | val ABSA-F1 0.8836 | 31s
  Fold 3 | epoch 9/12 | loss 0.1291 | val ABSA-F1 0.8841 | 31s
  Fold 3 | epoch 10/12 | loss 0.1128 | val ABSA-F1 0.8858 | 31s
  Fold 3 | epoch 11/12 | loss 0.0995 | val ABSA-F1 0.8874 | 31s
  Fold 3 | epoch 12/12 | loss 0.0922 | val ABSA-F1 0.8853 | 32s


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Fold 4 | epoch 1/12 | loss 1.1362 | val ABSA-F1 0.5777 | 32s
  Fold 4 | epoch 2/12 | loss 0.6966 | val ABSA-F1 0.6535 | 32s
  Fold 4 | epoch 3/12 | loss 0.4997 | val ABSA-F1 0.7922 | 32s
  Fold 4 | epoch 4/12 | loss 0.3757 | val ABSA-F1 0.8440 | 32s
  Fold 4 | epoch 5/12 | loss 0.2942 | val ABSA-F1 0.8568 | 32s
  Fold 4 | epoch 6/12 | loss 0.2347 | val ABSA-F1 0.8651 | 31s
  Fold 4 | epoch 7/12 | loss 0.1905 | val ABSA-F1 0.8712 | 32s
  Fold 4 | epoch 8/12 | loss 0.1576 | val ABSA-F1 0.8778 | 32s
  Fold 4 | epoch 9/12 | loss 0.1320 | val ABSA-F1 0.8794 | 31s
  Fold 4 | epoch 10/12 | loss 0.1132 | val ABSA-F1 0.8801 | 32s
  Fold 4 | epoch 11/12 | loss 0.1005 | val ABSA-F1 0.8816 | 31s
  Fold 4 | epoch 12/12 | loss 0.0933 | val ABSA-F1 0.8852 | 32s

OOF ABSA-F1 (ngưỡng 0,5): 0.8840 | sau chỉnh ngưỡng: 0.8839 (lạc quan vì ngưỡng chọn trên chính OOF)
HOLDOUT ABSA-F1: ngưỡng 0,5 = 0.8887 | ngưỡng chỉnh từ OOF = 0.8880   (TF-IDF cùng tập: 0,764)
          nhan  so_mau_that  so_du_doan     

In [ ]:
# ===== PhoBERT-large (seed 42) =====
SEED3 = 42
os.environ["PYTHONHASHSEED"] = str(SEED3)
random.seed(SEED3); np.random.seed(SEED3); torch.manual_seed(SEED3); torch.cuda.manual_seed_all(SEED3)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
torch.cuda.empty_cache()

summary3 = run_kfold("train.csv", "visobert_test_ids.csv", "test_blind.csv", out_dir=OUT_DIR,
                     tag="phobert_large",
                     model="vinai/phobert-large", n_folds=5,
                     epochs=8, batch_size=8,            # mô hình lớn hội tụ nhanh hơn, batch nhỏ cho vừa bộ nhớ T4
                     lr=2e-5, head_lr=1e-4,             # lr thấp hơn base để tránh mất ổn định
                     llrd=0.95,                         # 24 lớp nên giảm lr theo lớp nhẹ hơn (base dùng 0,9)
                     weight_decay=0.01, warmup=0.1, patience=2,
                     max_len=192,                       # ngắn hơn một chút để nhanh và đỡ tốn bộ nhớ
                     seed=SEED3, text_fn=segment, save_weights=False)

Thiết bị: cuda (Tesla T4)
Dev (K-fold): 3750 | holdout: 938


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Chia fold bằng: iterative-stratification


... (15 outputs left out)

  Fold 3 | epoch 1/8 | loss 0.9784 | val ABSA-F1 0.6830 | 113s
  Fold 3 | epoch 2/8 | loss 0.4755 | val ABSA-F1 0.8284 | 114s
  Fold 3 | epoch 3/8 | loss 0.2908 | val ABSA-F1 0.8678 | 113s
  Fold 3 | epoch 4/8 | loss 0.1937 | val ABSA-F1 0.8720 | 113s
  Fold 3 | epoch 5/8 | loss 0.1325 | val ABSA-F1 0.8841 | 113s
  Fold 3 | epoch 6/8 | loss 0.0906 | val ABSA-F1 0.8873 | 113s
  Fold 3 | epoch 7/8 | loss 0.0647 | val ABSA-F1 0.8912 | 114s
  Fold 3 | epoch 8/8 | loss 0.0515 | val ABSA-F1 0.8903 | 112s


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Fold 4 | epoch 1/8 | loss 0.9896 | val ABSA-F1 0.6549 | 114s
  Fold 4 | epoch 2/8 | loss 0.5042 | val ABSA-F1 0.8102 | 114s
  Fold 4 | epoch 3/8 | loss 0.3125 | val ABSA-F1 0.8618 | 114s
  Fold 4 | epoch 4/8 | loss 0.2093 | val ABSA-F1 0.8769 | 115s
  Fold 4 | epoch 5/8 | loss 0.1394 | val ABSA-F1 0.8735 | 116s
  Fold 4 | epoch 6/8 | loss 0.0934 | val ABSA-F1 0.8797 | 116s
  Fold 4 | epoch 7/8 | loss 0.0681 | val ABSA-F1 0.8809 | 115s
  Fold 4 | epoch 8/8 | loss 0.0538 | val ABSA-F1 0.8821 | 115s

OOF ABSA-F1 (ngưỡng 0,5): 0.8836 | sau chỉnh ngưỡng: 0.8846 (lạc quan vì ngưỡng chọn trên chính OOF)
HOLDOUT ABSA-F1: ngưỡng 0,5 = 0.8797 | ngưỡng chỉnh từ OOF = 0.8789   (TF-IDF cùng tập: 0,764)
          nhan  so_mau_that  so_du_doan     P     R    F1
nhan_vien_khen          494         500 0.960 0.972 0.966
 nhan_vien_che          120         103 0.922 0.792 0.852
  sach_se_khen          301         303 0.983 0.990 0.987
   sach_se_che           89          93 0.860 0.899 0.879
   vi_tri